In [1]:
# Dependências (execute esta célula se estiver em um ambiente novo)
# Objetivo: garantir que o OpenCV esteja disponível no MESMO kernel do notebook.
import importlib
import sys
import subprocess

def _ensure_import(module_name: str, pip_name: str | None = None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        pkg = pip_name or module_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        return importlib.import_module(module_name)

# Necessário para este notebook
_ensure_import("cv2", "opencv-python")
_ensure_import("numpy")
_ensure_import("PIL", "Pillow")

# Opcional: se falhar no Windows, instale em um venv/conda ou habilite Long Paths
try:
    importlib.import_module("ipywidgets")
except ImportError:
    print("Aviso: ipywidgets não está instalado neste kernel.\n"
          "Se você precisar da interface interativa, instale ipywidgets no mesmo ambiente do Jupyter.")


## Seção Prática

In [ ]:
# Instalação de dependências (execute 1x neste kernel)
import sys
import subprocess
from pathlib import Path

req_file = Path("requirements.txt")

if req_file.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_file)])
else:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "opencv-python",
            "numpy",
            "Pillow",
            "ipywidgets",
        ]
    )

print("Dependências instaladas neste kernel.")


### Bibliotecas utilizadas

- **OpenCV (`cv2`)**: leitura de imagem, redimensionamento e (principalmente) o remapeamento de pixels com `cv2.remap`.
- **NumPy (`numpy`)**: criação das grades de coordenadas e cálculos matriciais do mapa de profundidade e dos deslocamentos.
- **PIL (`Pillow`)** + **IPython display**: conversão para RGB e exibição da imagem no notebook.
- **ipywidgets**: criação da interface interativa com sliders e `widgets.interact`.

In [8]:
import cv2
import numpy as np
from IPython.display import display
from PIL import Image
import ipywidgets as widgets

### Carregamento da imagem

 notebook define um caminho em `caminho_da_imagem` e tenta ler com:

- `img = cv2.imread(caminho_da_imagem)`

In [9]:
caminho_da_imagem = 'C:\\Users\\vinic\\OneDrive\\Documentos\\ReposMackenzie\\imagens-pseudo3D\\cachorro-gravata.jpg' 
img = cv2.imread(caminho_da_imagem)

if img is None:
    raise ValueError("A imagem não foi encontrada.")

### Ajuste de tamanho (redimensionamento)

Depois de obter `altura` e `largura`, o notebook limita a imagem a:

- `max_width = 600`
- `max_height = 500`

Se a imagem for maior que esses limites, calcula uma **escala** e redimensiona mantendo a proporção:

- `escala = min(max_width / largura, max_height / altura)`
- `cv2.resize(...)`

In [ ]:
altura, largura = img.shape[:2]

max_width, max_height = 600, 500
if largura > max_width or altura > max_height:
    escala = min(max_width / largura, max_height / altura)
    img = cv2.resize(img, (int(largura * escala), int(altura * escala)))
    altura, largura = img.shape[:2]

### Criação de um “mapa de profundidade” sintético

Como não há um depth map real (estimado por rede neural, estéreo, etc.), o notebook cria um mapa **artificial** baseado na distância ao centro da imagem:

1. cria grades de coordenadas:
   - `y, x = np.mgrid[0:altura, 0:largura]`
2. calcula o centro:
   - `centro_x = largura / 2`, `centro_y = altura / 2`
3. define um mapa com formato “colina” (tipo gaussiano/exponencial radial):
   - `mapa_profundidade = exp(-dist2 / escala)`

Em seguida, converte esse mapa em deslocamento máximo de pixels:

- `forca_maxima = 30`
- `profundidade_calculada = mapa_profundidade * forca_maxima`

Na prática, isso faz o **centro deslocar mais** e as bordas deslocarem menos (ou vice‑versa, dependendo do sinal do controle), produzindo a sensação de profundidade ao mover.

In [13]:
y, x = np.mgrid[0:altura, 0:largura]
centro_x, centro_y = largura / 2, altura / 2
mapa_profundidade = np.exp(-((x - centro_x)**2 + (y - centro_y)**2) / (0.1 * (largura**2 + altura**2)))

forca_maxima = 30 
profundidade_calculada = mapa_profundidade * forca_maxima

print(f"Dimensões da imagem: {largura}x{altura}\n")

Dimensões da imagem: 300x168



### Interação, efeito pseudo-3D e resultado final

O notebook define a função `exibir_simulacao(...)` com dois parâmetros controlados por sliders:

- `dx`: varia de `-1.0` a `+1.0` (movimento horizontal)
- `dy`: varia de `-1.0` a `+1.0` (movimento vertical)

Ela é conectada à UI com:

- `widgets.interact(exibir_simulacao)`

Cada alteração no slider chama a função novamente e atualiza a imagem mostrada.

Dentro de `exibir_simulacao`, o notebook calcula mapas de remapeamento (para cada pixel de saída, de onde vem o pixel na imagem original):

- `mapa_x = x - profundidade_calculada * dx`
- `mapa_y = y - profundidade_calculada * dy`

E aplica o remapeamento:

- `imagem_deslocada = cv2.remap(img, mapa_x, mapa_y, interpolation=..., borderMode=...)`

Configurações importantes:

- **`interpolation=cv2.INTER_LINEAR`**: suaviza a amostragem ao deslocar.
- **`borderMode=cv2.BORDER_REPLICATE`**: quando o deslocamento “puxa” pixels de fora da imagem, repete a borda para evitar áreas pretas.

Esse processo é um uso típico de **DIBR (Depth‑Image‑Based Rendering)** simplificado: em vez de renderizar um modelo 3D, desloca-se a imagem conforme uma profundidade por pixel.

Como o OpenCV trabalha em **BGR**, o notebook converte para **RGB** antes de mostrar:

- `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`
- `Image.fromarray(...)`
- `display(...)`

Ele também imprime os valores atuais do “joystick” (`dx`, `dy`) para feedback.

In [ ]:
def exibir_simulacao(dx=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='X (←→):'),
                     dy=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0, description='Y (↑↓):')):
    
    # Calcula os mapas de deslocamento
    mapa_x = np.float32(x - profundidade_calculada * dx)
    mapa_y = np.float32(y - profundidade_calculada * dy)
    
    # Aplica o efeito DIBR
    imagem_deslocada = cv2.remap(img, mapa_x, mapa_y, 
                                 interpolation=cv2.INTER_LINEAR, 
                                 borderMode=cv2.BORDER_REPLICATE)
    
    # Converte para RGB
    img_rgb = cv2.cvtColor(imagem_deslocada, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)

    display(img_pil)
    
    # Mostra informações do joystick
    print(f"Joystick → X: {dx:+.2f} | Y: {dy:+.2f}")

# Cria a interface interativa com @interact
widgets.interact(exibir_simulacao)